In [ ]:
import pandas as pd

caminho_arquivo = "dados_saneamento/fortaleza_dados_saneamento.xlsx"

# lê o excel
df_saneamento_dados = pd.read_excel(
    caminho_arquivo,
    header=2
)

# pega as linhas desejadas
df_saneamento_formatado = df_saneamento_dados.iloc[29:37]

# transpõe a tabela
df_saneamento_formatado = df_saneamento_formatado.T

# primeira linha vira cabeçalho
df_saneamento_formatado.columns = (
    df_saneamento_formatado.iloc[0]
)

# remove linha usada no cabeçalho
df_saneamento_formatado = (
    df_saneamento_formatado.iloc[1:]
    .reset_index()
)

df_saneamento_formatado = (
    df_saneamento_formatado.rename(
        columns={
            "index": "Ano",
            "População total que mora em domicílios com acesso ao serviço de coleta de esgoto (pessoas) (SNIS/SINISA)": "Pop_total_com_coleta_esgoto",
            "População total que mora em domicílios sem acesso ao serviço de coleta de esgoto (pessoas) (SNIS/SINISA)": "Pop_sem_coleta_esgoto",
            "Parcela da população total que mora em domicílios com acesso ao serviço de coleta de esgoto (% da população) (SNIS/SINISA)": "Perc_pop_total_com_coleta_esgoto",
            "Parcela da população total que mora em domicílios sem acesso ao serviço de coleta de esgoto (% da população) (SNIS/SINISA)": "Perc_sem_coleta_esgoto",
            "População urbana que mora em domicílios com acesso ao serviço de coleta de esgoto (pessoas) (SNIS/SINISA)": "Pop_urbana_com_coleta_esgoto",
            "População urbana que mora em domicílios sem acesso ao serviço de coleta de esgoto (pessoas) (SNIS/SINISA)": "Pop_urbana_sem_coleta_esgoto",
            "Parcela da população urbana que mora em domicílios com acesso ao serviço de coleta de esgoto (% da população) (SNIS/SINISA)": "Perc_pop_urbana_com_coleta_esgoto",
            "Parcela da população urbana que mora em domicílios sem acesso ao serviço de coleta de esgoto (% da população) (SNIS/SINISA)": "Perc_pop_urbana_sem_coleta_esgoto",               
        }
    )
)

colunas_percentuais = [
    "Perc_pop_total_com_coleta_esgoto",
    "Perc_sem_coleta_esgoto",
    "Perc_pop_urbana_com_coleta_esgoto",
    "Perc_pop_urbana_sem_coleta_esgoto"
]

print(
    df_saneamento_formatado[
        colunas_percentuais
    ].head()
)


for coluna in colunas_percentuais:
    df_saneamento_formatado[coluna] = (
        pd.to_numeric(
            df_saneamento_formatado[coluna],
            errors="coerce"
        ).round(4)
    )


# substituir - por nao_informado
df_saneamento_formatado = (
   df_saneamento_formatado
    .replace("-", "nao_informado")
)

# drop nas colunas irrelevantes
colunas_drop = [
    "Pop_total_com_coleta_esgoto",
    "Perc_pop_total_com_coleta_esgoto",
    "Perc_pop_urbana_com_coleta_esgoto",
    "Perc_pop_urbana_sem_coleta_esgoto",
    "Pop_urbana_com_coleta_esgoto",
    "Pop_urbana_sem_coleta_esgoto",
]

df_saneamento_formatado = df_saneamento_formatado.drop(
    columns=colunas_drop
)

meses = range(1, 13)

novas_linhas = []

for _, linha in df_saneamento_formatado.iterrows():
    for mes in meses:
        novas_linhas.append({
            "Ano_mes": f"{linha['Ano']}-{mes:02d}",
            "Pop_sem_coleta_esgoto": linha["Pop_sem_coleta_esgoto"],
            "Perc_sem_coleta_esgoto": linha["Perc_sem_coleta_esgoto"]
        })

df_saneamento_mensal  = pd.DataFrame(novas_linhas)

"""
Ano e mes,
População total que mora em domicílios sem acesso ao serviço de coleta de esgoto (pessoas) (SNIS/SINISA),
Parcela da população total que mora em domicílios sem acesso ao serviço de coleta de esgoto (% da população) (SNIS/SINISA),
"""

print(df_saneamento_formatado.isnull().sum())

print("Arquivo CSV salvo com sucesso!")


Indicador Perc_pop_total_com_coleta_esgoto Perc_sem_coleta_esgoto  \
0                                    0.642                  0.358   
1                                    0.665                  0.335   
2                                    0.628                  0.372   
3                                    0.559                  0.441   
4                                    0.553                  0.447   

Indicador Perc_pop_urbana_com_coleta_esgoto Perc_pop_urbana_sem_coleta_esgoto  
0                                     0.642                             0.358  
1                                     0.665                             0.335  
2                                         -                                 -  
3                                     0.559                             0.441  
4                                     0.553                             0.447  
Indicador
Ano                       0
Pop_sem_coleta_esgoto     0
Perc_sem_coleta_esgoto    0
dtype: int6

Pegando somente os dados de 2017 a 2024 e ordenando do menor para o maior

In [11]:
# Pegando os Ano_mes de 2017 a 2024
datas = pd.date_range(start="2017-01-01", end="2024-12-31", freq="MS")
df_temporal = pd.DataFrame({'Ano_mes' : datas})
df_temporal['Ano'] = df_temporal['Ano_mes'].dt.year

df_saneamento_final = pd.merge(
    df_temporal, df_saneamento_formatado[['Ano', 'Pop_sem_coleta_esgoto', 'Perc_sem_coleta_esgoto']], 
    on='Ano', how='left'
    )

print(df_saneamento_final.head(15))

#Ordenando por Ano_mes
df_saneamento_formatado = df_saneamento_formatado.sort_values("Ano").reset_index(drop=True)

print(df_saneamento_formatado[["Ano", "Pop_sem_coleta_esgoto", "Perc_sem_coleta_esgoto"]].head())


      Ano_mes   Ano Pop_sem_coleta_esgoto  Perc_sem_coleta_esgoto
0  2017-01-01  2017               1294695                   0.493
1  2017-02-01  2017               1294695                   0.493
2  2017-03-01  2017               1294695                   0.493
3  2017-04-01  2017               1294695                   0.493
4  2017-05-01  2017               1294695                   0.493
5  2017-06-01  2017               1294695                   0.493
6  2017-07-01  2017               1294695                   0.493
7  2017-08-01  2017               1294695                   0.493
8  2017-09-01  2017               1294695                   0.493
9  2017-10-01  2017               1294695                   0.493
10 2017-11-01  2017               1294695                   0.493
11 2017-12-01  2017               1294695                   0.493
12 2018-01-01  2018               1324454                   0.501
13 2018-02-01  2018               1324454                   0.501
14 2018-03

Exportando os dados em csv

In [12]:
# salva em csv
df_saneamento_final.to_csv(
    "dados_formatados/SANEAMENTO_DADOS_FORTALEZA.csv",
    index=False,
    encoding="utf-8-sig",
    float_format="%.3f"
)